# Study 1 smoke test: NIH ChestX-ray14, CheXzero and CLEAR

Goal of this notebook (week 1 go/no-go check):
1. Confirm the NIH data and official split files are readable, and count test images by sex, age band and view.
2. Load CheXzero (one checkpoint) and CLEAR, score 100 NIH test images on the 14 findings, and record speed and GPU memory.
3. Check that CLEAR's concept assets load within Kaggle memory.

Settings before running: **Accelerator = GPU (T4 or P100)**, **Internet = On**, and the dataset **NIH Chest X-rays** (`nih-chest-xrays/data`) added as input.
Then use **Run All**. At the end, copy the printed SMOKE TEST SUMMARY block and send it back.

Do not look at AUROC here: 100 images is far too few, and the test set should only be scored in full after the OSF pre-registration is posted.

In [ ]:
# 1. Environment check
import sys, subprocess, torch, platform
print("Python", platform.python_version(), "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU in the notebook settings."

In [ ]:
# 2. Install packages (CLIP for CheXzero, the CLEAR library, gdown for the CheXzero weights)
!pip install -q git+https://github.com/openai/CLIP.git gdown
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!pip install -q /kaggle/working/CLEAR

In [ ]:
# 3. Locate the NIH files (works whatever the input folder is called)
import glob, os
import pandas as pd, numpy as np

def find_one(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found. Is the NIH Chest X-rays dataset added as input?"
    return hits[0]

entry_csv = find_one("Data_Entry_2017.csv")
test_list = find_one("test_list.txt")
trainval_list = find_one("train_val_list.txt")
img_paths = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/images*/**/*.png", recursive=True)}
print("Images found:", len(img_paths))       # expect 112,120

In [ ]:
# 4. Metadata, split and subgroup counts for the official test split (no model involved)
df = pd.read_csv(entry_csv)
df = df.rename(columns={"Image Index":"image","Finding Labels":"labels","Patient ID":"patient",
                        "Patient Age":"age","Patient Gender":"sex","View Position":"view"})
test_ids = set(open(test_list).read().split())
tv_ids = set(open(trainval_list).read().split())
df["split"] = np.where(df.image.isin(test_ids), "test", np.where(df.image.isin(tv_ids), "train_val", "none"))

FINDINGS = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
for f in FINDINGS:
    df[f] = df.labels.str.contains(f, regex=False).astype(int)

# leakage check: no patient should appear in both splits
overlap = set(df[df.split=="test"].patient) & set(df[df.split=="train_val"].patient)
print("Patients in both train_val and test:", len(overlap))

print("Ages over 100 (to exclude):", int((df.age > 100).sum()))
t = df[(df.split=="test") & (df.age <= 100)].copy()
t["age_band"] = pd.cut(t.age, [-1, 18, 40, 60, 80, 200], labels=["0-18","18-40","40-60","60-80","80+"])
print("Test images:", len(t), "| test patients:", t.patient.nunique())
print(pd.crosstab(t.age_band, t.sex, margins=True))
print("\nAP share by age band:")
print(t.groupby("age_band", observed=True).view.apply(lambda v: round((v=="AP").mean(), 3)))
pos = t.groupby("sex")[FINDINGS].sum().T
pos2 = t.groupby("age_band", observed=True)[FINDINGS].sum().T
print("\nPositive test images per finding, by sex and by age band:")
print(pd.concat([pos, pos2], axis=1))
pos_table = pd.concat([pos, pos2], axis=1)
pos_table.to_csv("/kaggle/working/test_positives_by_subgroup.csv")

In [ ]:
# 5. Pick 100 test images for the smoke test (fixed seed)
sample = t.sample(100, random_state=2026).reset_index(drop=True)
sample["path"] = sample.image.map(img_paths)
assert sample.path.notna().all()
PROMPTS = {"Atelectasis":"atelectasis","Cardiomegaly":"cardiomegaly","Effusion":"pleural effusion",
           "Infiltration":"infiltration","Mass":"mass","Nodule":"nodule","Pneumonia":"pneumonia",
           "Pneumothorax":"pneumothorax","Consolidation":"consolidation","Edema":"edema",
           "Emphysema":"emphysema","Fibrosis":"fibrosis","Pleural_Thickening":"pleural thickening",
           "Hernia":"hernia"}

In [ ]:
# 6. Download the CheXzero checkpoints from the authors' Google Drive folder
import gdown
os.makedirs("/kaggle/working/chexzero_weights", exist_ok=True)
try:
    gdown.download_folder("https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
                          output="/kaggle/working/chexzero_weights", quiet=True, remaining_ok=True)
except Exception as e:
    print("Google Drive download failed:", e)
    print("Fallback: download the folder on your computer and add it to Kaggle as a private dataset.")
ckpts = sorted(glob.glob("/kaggle/working/chexzero_weights/**/*.pt", recursive=True))
print(len(ckpts), "checkpoints:", [os.path.basename(c) for c in ckpts])

In [ ]:
# 7. CheXzero zero-shot on the 100 images (one checkpoint; the full run will use the ensemble)
import time, clip
from PIL import Image
from torchvision.transforms import Compose, Normalize, Resize, InterpolationMode
dev = "cuda"
results = {}

def pair_scores(img_feat, pos_feat, neg_feat):
    # softmax over the positive and negative prompt, as in CheXzero's run_softmax_eval
    p = img_feat @ pos_feat.T
    n = img_feat @ neg_feat.T
    return (torch.exp(p) / (torch.exp(p) + torch.exp(n))).cpu().numpy()

if ckpts:
    cz, _ = clip.load("ViT-B/32", device="cpu", jit=False)
    cz.load_state_dict(torch.load(ckpts[0], map_location="cpu"))
    cz = cz.to(dev).eval()
    tf = Compose([Normalize((101.48761,)*3, (83.43944,)*3),
                  Resize(224, interpolation=InterpolationMode.BICUBIC)])
    def cz_img(p):
        a = np.asarray(Image.open(p).convert("L").resize((320, 320), Image.BICUBIC), dtype=np.float32)
        return tf(torch.from_numpy(np.repeat(a[None], 3, axis=0)))
    with torch.no_grad():
        labels = [PROMPTS[f] for f in FINDINGS]
        def txt(tmpl):
            e = cz.encode_text(clip.tokenize([tmpl.format(l) for l in labels], context_length=77).to(dev)).float()
            return e / e.norm(dim=-1, keepdim=True)
        pos_f, neg_f = txt("{}"), txt("no {}")
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
        out = []
        for i in range(0, len(sample), 25):
            x = torch.stack([cz_img(p) for p in sample.path[i:i+25]]).to(dev)
            f = cz.encode_image(x).float(); f = f / f.norm(dim=-1, keepdim=True)
            out.append(pair_scores(f, pos_f, neg_f))
        torch.cuda.synchronize()
        results["chexzero"] = {"sec_per_image": round((time.time()-t0)/len(sample), 4),
                               "peak_gpu_gb": round(torch.cuda.max_memory_allocated()/1e9, 2)}
    cz_scores = pd.DataFrame(np.concatenate(out), columns=FINDINGS); cz_scores.insert(0, "image", sample.image)
    cz_scores.to_csv("/kaggle/working/smoke_chexzero_scores.csv", index=False)
    del cz; torch.cuda.empty_cache()
    print(results["chexzero"]); print(cz_scores.describe().loc[["min","mean","max"]].round(3))

In [ ]:
# 8. CLEAR zero-shot on the same 100 images (weights download from Hugging Face on first use)
from clear import load_pretrained, tokenize
try:
    model, preprocess = load_pretrained(device=dev); model.eval()
    with torch.inference_mode():
        labels = [PROMPTS[f] for f in FINDINGS]
        def ctxt(tmpl):
            e = model.encode_text(tokenize([tmpl.format(l) for l in labels]).to(dev)).float()
            return e / e.norm(dim=-1, keepdim=True)
        pos_f, neg_f = ctxt("{}"), ctxt("no {}")
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
        out = []
        for i in range(0, len(sample), 25):
            x = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in sample.path[i:i+25]]).to(dev)
            f = model.encode_image(x).float(); f = f / f.norm(dim=-1, keepdim=True)
            out.append(pair_scores(f, pos_f, neg_f))
        torch.cuda.synchronize()
        results["clear"] = {"sec_per_image": round((time.time()-t0)/len(sample), 4),
                            "peak_gpu_gb": round(torch.cuda.max_memory_allocated()/1e9, 2),
                            "input_shape": list(x.shape[1:])}
    cl_scores = pd.DataFrame(np.concatenate(out), columns=FINDINGS); cl_scores.insert(0, "image", sample.image)
    cl_scores.to_csv("/kaggle/working/smoke_clear_scores.csv", index=False)
    del model; torch.cuda.empty_cache()
    print(results["clear"]); print(cl_scores.describe().loc[["min","mean","max"]].round(3))
except Exception as e:
    results["clear"] = {"error": repr(e)[:500]}
    print("CLEAR failed:", e)

In [ ]:
# 9. CLEAR concept assets: run the authors' quick-start with concept output on one image.
# This downloads the concept files (over 1 GB) and is the memory check from the plan.
import resource
one = sample.path[0]
t0 = time.time()
r = subprocess.run([sys.executable, "/kaggle/working/CLEAR/examples/quickstart.py", one, "--top-concepts", "10"],
                   capture_output=True, text=True, cwd="/kaggle/working/CLEAR")
results["clear_concepts"] = {"returncode": r.returncode, "seconds": round(time.time()-t0, 1),
                             "child_peak_ram_gb": round(resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss/1e6, 2)}
print(r.stdout[-3000:]); print(r.stderr[-2000:])

In [ ]:
# 10. Summary to send back
import json
summary = {
  "images_found": len(img_paths),
  "test_images_age<=100": int(len(t)), "test_patients": int(t.patient.nunique()),
  "ages_over_100_all_splits": int((df.age > 100).sum()),
  "patients_in_both_splits": len(overlap),
  "test_by_sex": t.sex.value_counts().to_dict(),
  "test_by_age_band": {str(k): int(v) for k, v in t.age_band.value_counts().sort_index().items()},
  "test_AP_share": round(float((t.view=="AP").mean()), 3),
  "chexzero_checkpoints": len(ckpts),
  **results,
}
if "chexzero" in results and "error" not in results.get("clear", {"error":1}):
    summary["score_corr_chexzero_vs_clear"] = round(float(np.corrcoef(cz_scores[FINDINGS].values.ravel(),
                                                                      cl_scores[FINDINGS].values.ravel())[0,1]), 3)
est = {k: round(v["sec_per_image"]*25596/3600, 2) for k, v in results.items() if "sec_per_image" in v}
summary["est_hours_full_test_split"] = est
print("===== SMOKE TEST SUMMARY =====")
print(json.dumps(summary, indent=1, default=str))
print("===== END =====")